In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
import pandas as pd

# --- Load Tables ---

# CineClub gifting table (gift card purchases & redemptions)
df_gifting = spark.table("cpx_dataanalytics_silver.cineclub.gifting")

# CineClub subscriber profile snapshot
df_profile = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot").filter(F.col("snapshot_date_id") == 20260721)

# Master transaction table (box office + concessions, 2021-present)
df_transactions = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")

print("Tables loaded successfully.")
print(f"  gifting rows:  {df_gifting.count():,}")
print(f"  profile rows:  {df_profile.count():,}")
print("  transactions:  (large table - count skipped)")

In [0]:
# Tim Horton's gift codes for lookup
gift_codes = [
    "GIFT12C-KQHQ-4FT4-BMRH-MP3Q",
    "GIFT12C-MFUU-E2D6-AVBP-D7VK",
    "GIFT12C-8PVD-KYVM-MCWK-LW1P",
    "GIFT12C-UL42-X45F-4PGX-P38X",
    "GIFT12C-XTVE-N0G7-V6K4-1109",
    "GIFT12C-KQ3A-E86L-V8C4-JKKV",
    "GIFT12C-Y93U-XNKX-PD8T-QURU",
    "GIFT12C-872Q-C5N1-CCNJ-BBVQ",
    "GIFT12C-LHTD-Q43H-QH9X-URME",
    "GIFT12C-JR5B-A7Q1-DC3K-FUCL",
    "GIFT12C-XDVF-88YJ-7UNH-BXEX",
    "GIFT12C-B5YD-3F1M-YDUK-BDEH",
    "GIFT12C-N5N7-YFB2-X1C4-71V2",
    "GIFT12C-T23B-KNDM-AUJB-TUKX",
    "GIFT12C-G89Q-C2WD-EXBH-YYCD",
    "GIFT12C-6J7C-QWYQ-NMWQ-9UV9",
]

print(f"{len(gift_codes)} gift codes loaded")


In [0]:
# Filter gifting table for our Tim Horton's codes and get redeemer info
df_tims_redeemers = (
    df_gifting
    .filter(F.col("id").isin(gift_codes))
    .select(
        F.col("id").alias("gift_code"),
        "redeemer_id",
        F.to_date("redemption_date").alias("redemption_date")
    )
    .orderBy("redemption_date")
)

display(df_tims_redeemers)

In [0]:
# Join redeemers to profile snapshot on redeemer_id = cc_user_profile_id
df_redeemer_profile = (
    df_tims_redeemers
    .join(
        df_profile.select(
            "cc_user_profile_id",
            "plan_id",
            "renewal_status",
            F.col("recognitions_left").alias("unredeemed_tickets")
        ),
        df_tims_redeemers["redeemer_id"] == df_profile["cc_user_profile_id"],
        how="left"
    )
    .drop("cc_user_profile_id")
    .orderBy("redemption_date")
)

display(df_redeemer_profile)

In [0]:
# Get cde_id for each redeemer via profile (redeemer_id = cc_user_profile_id)
df_redeemer_cde = (
    df_tims_redeemers
    .join(
        df_profile.select("cc_user_profile_id", "cde_id"),
        df_tims_redeemers["redeemer_id"] == df_profile["cc_user_profile_id"],
        how="inner"
    )
    .select("redeemer_id", "gift_code", "redemption_date", F.col("cde_id").cast("int").alias("cde_id"))
)

# Join transactions, filtering to ±1 year from each member's redemption date
df_redeemer_txns = (
    df_redeemer_cde
    .join(
        df_transactions.select(
            "CDE_ID", "Transaction_DateTime", "Transaction_Number",
            "VisitDateId", "TicketFLAG", "ConcessionFLAG", "CineClubFLAG",
            "Net_Spend", "Net_Revenue", "Quantity", "LocationId", "film_id", "CineClub_Discount_Type_ID"
        ),
        df_redeemer_cde["cde_id"] == df_transactions["CDE_ID"],
        how="inner"
    )
    .filter(
        (F.to_date("Transaction_DateTime") >= F.date_sub(F.col("redemption_date"), 365)) &
        (F.to_date("Transaction_DateTime") <= F.date_add(F.col("redemption_date"), 365))
    )
    .withColumn("transaction_date", F.to_date("Transaction_DateTime"))
    .withColumn("period", F.when(F.col("transaction_date") < F.col("redemption_date"), "pre").otherwise("post"))
    .drop("CDE_ID")
)

# # print(f"Total transactions (±1yr): {df_redeemer_txns.count():,}")
# # print(f"Distinct redeemers with txns: {df_redeemer_txns.select('redeemer_id').distinct().count()}")
# display(df_redeemer_txns.orderBy("redeemer_id", "transaction_date"))

In [0]:
# Load film dimension for title_id mapping
df_film = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film").select("film_id", "title_id")

# Load location dimension for location type
df_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location").select(
    F.col("LocationID").alias("loc_id"),
    "LC_Location_Type_Description"
)

# Enrich transactions with title_id and location type
df_txns_enriched = (
    df_redeemer_txns
    .join(df_film, on="film_id", how="left")
    .join(df_location, df_redeemer_txns["LocationId"] == df_location["loc_id"], how="left")
    .drop("loc_id")
)

# Helper: visit key
visit_key = F.concat_ws("|", F.col("VisitDateId").cast("string"), F.col("LocationId").cast("string"))

# Aggregate behavioral metrics per redeemer, split by pre/post
df_behavior = (
    df_txns_enriched
    .groupBy("redeemer_id", "period")
    .agg(
        # 1. Unique titles watched
        F.countDistinct("title_id").alias("unique_titles"),

        # 2. Total visits (unique VisitDateId + LocationId per customer)
        F.countDistinct(visit_key).alias("total_visits"),

        # 2a. Theatre visits
        F.countDistinct(
            F.when(F.col("LC_Location_Type_Description").like("%Theatre%"), visit_key)
        ).alias("theatre_visits"),

        # 2b. Restaurant visits
        F.countDistinct(
            F.when(F.col("LC_Location_Type_Description").like("%Restaurant%"), visit_key)
        ).alias("restaurant_visits"),

        # 3. CineClub visits
        F.countDistinct(
            F.when(F.col("CineClubFLAG") == "Yes", visit_key)
        ).alias("cineclub_visits"),

        # 4. Tickets purchased
        F.sum(F.when(F.col("TicketFLAG") == "Yes", F.col("Quantity")).otherwise(0)).alias("tickets_purchased"),

        # 4. Regular Ticket Revenue
        F.sum(F.when((F.col("TicketFLAG") == "Yes") & ((F.col("CineClubFLAG") == "No") | (F.col("CineClub_Discount_Type_ID") == 1)), F.col("Net_Revenue")).otherwise(0)).alias("regular_ticket_revenue"),

        # 4. CC Ticket Revenue (member priced + free ticket redemptions)
        F.sum(F.when((F.col("TicketFLAG") == "Yes") & (F.col("CineClub_Discount_Type_ID").isin(5, 6)), F.col("Net_Revenue")).otherwise(0)).alias("CC_ticket_revenue"),
      
        # 5. Member Priced Tickets (Discount Type ID = 5)
        F.sum(F.when((F.col("TicketFLAG") == "Yes") & (F.col("CineClub_Discount_Type_ID") == 5), F.col("Quantity")).otherwise(0)).alias("member_priced_tickets"),

        # 6. Free tickets redeemed (Discount Type ID = 6)
        F.sum(F.when((F.col("TicketFLAG") == "Yes") & (F.col("CineClub_Discount_Type_ID") == 6), F.col("Quantity")).otherwise(0)).alias("free_tickets_redeemed"),

        # 7. Concession Revenue - Total
        F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)).alias("concession_revenue_total"),

        # 7a. Concession Revenue - CineClub visits
        F.sum(F.when((F.col("ConcessionFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes"), F.col("Net_Revenue")).otherwise(0)).alias("concession_revenue_cc"),

        # 7b. Concession Revenue - Non-CineClub visits
        F.sum(F.when((F.col("ConcessionFLAG") == "Yes") & (F.col("CineClubFLAG") != "Yes"), F.col("Net_Revenue")).otherwise(0)).alias("concession_revenue_non_cc"),
    )
    .orderBy("redeemer_id", "period")
)


In [0]:
display(df_behavior)